# Chapter 4: Compactness

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 4, printed pp. 75-114; PDF pp. 90-129. Sections: 4.1-4.7.

    ## Chapter Goal

    Energy, bubbling, mean value and isoperimetric inequalities, removal of singularities, convergence modulo bubbling, and connected bubble trees.

    The guiding question is:

    > What kinds of limits are allowed when a sequence of J-holomorphic curves refuses to converge smoothly?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| energy | integral density on the domain | total mass conservation |
| bubble | localized energy packet | minimum quantum threshold |
| mean value inequality | local sup controlled by local average | scale-normalized bound |
| removal of singularities | finite energy puncture | extension check |
| bubble tree | graph of components and nodes | connected limit |


## Standalone Reading Guide

    Compactness is the counterweight to transversality. A regular moduli space can still be useless for counting if sequences run away. The chapter explains that escape has a controlled form: energy concentrates, scales are rescaled, and nonconstant bubbles appear. The limit is not usually a single smooth curve but a connected configuration carrying the same total homology class and energy.

The analytic estimates in the chapter are designed to prevent invisible loss. A mean value inequality turns small local energy into derivative control. An isoperimetric inequality controls short loops and helps analyze neck regions. Removal of singularities says that a puncture with finite energy is not a mysterious endpoint; after the right coordinate change, the curve extends. These facts combine into convergence modulo bubbling.

The notebook model treats energy as a probability density on a square domain. As a concentration parameter grows, mass collects near prescribed points. The check records that total energy is conserved and that extracted bubbles exceed a threshold. This is a toy model, but it mirrors the proof strategy: locate concentration, rescale, extract a nontrivial limit, and account for every unit of energy.

    ## Topics In This Notebook

    - energy identities and lower bounds for nonconstant bubbles
- gradient concentration as the visible trigger for bubbling
- mean value and isoperimetric estimates as analytic controls
- removable singularities and extension across punctures
- bubble trees and connectedness of the limiting configuration

    ## Visualization Storyboard

    - An energy-density plot shows concentration moving from a smooth base curve into quantized bubbles.
- A proof graph connects analytic estimates to Gromov compactness.
- A ledger checks energy conservation and the minimum bubble threshold.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'chapter-04'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['energy', 'bubble', 'mean value inequality', 'removal of singularities', 'bubble tree']
CONCEPT_EDGES = [('energy', 'bubble'), ('bubble', 'mean value inequality'), ('mean value inequality', 'removal of singularities'), ('removal of singularities', 'bubble tree')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=21, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Compactness')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '75-114',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Compactness. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
n = 160
x = np.linspace(-1, 1, n)
y = np.linspace(-1, 1, n)
X, Y = np.meshgrid(x, y)
centers = [(-0.35, -0.1), (0.42, 0.28)]
sigmas = [0.34, 0.12]
weights = [0.55, 0.45]
density = np.zeros_like(X)
for (cx, cy), sigma, weight in zip(centers, sigmas, weights):
    bump = np.exp(-((X - cx) ** 2 + (Y - cy) ** 2) / (2 * sigma**2))
    bump /= bump.sum()
    density += weight * bump
density /= density.sum()
threshold = 0.18
near_bubbles = []
for cx, cy in centers:
    mask = (X - cx) ** 2 + (Y - cy) ** 2 < 0.18**2
    near_bubbles.append(float(density[mask].sum()))

fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(density, extent=[-1, 1, -1, 1], origin="lower", cmap="viridis")
for cx, cy in centers:
    ax.scatter([cx], [cy], s=90, facecolor="none", edgecolor="white", linewidth=2)
ax.set_title("Energy concentration and candidate bubbles")
ax.set_xlabel("domain x")
ax.set_ylabel("domain y")
fig.colorbar(im, ax=ax, label="normalized energy density")
fig_path = save_matplotlib(fig, UNIT, "figures", "energy-concentration-bubbles.png")
plt.close(fig)

check = {
    "total_energy": float(density.sum()),
    "bubble_window_energies": near_bubbles,
    "threshold": threshold,
    "detected_bubbles": int(sum(e > threshold for e in near_bubbles)),
    "passed": bool(abs(float(density.sum()) - 1.0) < 1e-10 and sum(e > threshold for e in near_bubbles) >= 1),
}
check_path = save_json(check, UNIT, "checks", "energy-quantization-checks.json")
display_artifact(fig_path, width=620)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'energy', 'computational_object': 'integral density on the domain', 'check': 'total mass conservation'}, {'item': 'bubble', 'computational_object': 'localized energy packet', 'check': 'minimum quantum threshold'}, {'item': 'mean value inequality', 'computational_object': 'local sup controlled by local average', 'check': 'scale-normalized bound'}, {'item': 'removal of singularities', 'computational_object': 'finite energy puncture', 'check': 'extension check'}, {'item': 'bubble tree', 'computational_object': 'graph of components and nodes', 'check': 'connected limit'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Move the bubble centers or raise the threshold. The conservation check should remain true, while the detected bubble count changes exactly when a packet falls below the declared quantum.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - Bubbling is controlled failure of smooth convergence, not arbitrary pathology.
- Energy quantization makes compactification finite enough for moduli theory.
- Neck and removable-singularity estimates explain how bubble components attach.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'energy-concentration-bubbles.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'energy-quantization-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'energy-quantization-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
